# [Lab 2](https://github.com/hanggrian/IIT-CS587/blob/assets/assignments/lab2.pdf): LangGraph Workbench

The host machine for all labs is an EndeavourOS with Arch Linux as the base system.

## Requirements

1.  [x] Install Python version 3.10: [python310<sup>AUR</sup>](https://aur.archlinux.org/packages/python310)
1.  [x] Install VSCode: [visual-studio-code-bin<sup>AUR</sup>](https://aur.archlinux.org/packages/visual-studio-code-bin)
1.  [x] Check Python version.
    ```sh
    python --version
    ```
1.  [x] Create a virtual environment.
    ```sh
    source .venv/bin/activate
    ```
1.  Read [LangGraph documentation](https://langchain-ai.github.io/langgraph/).
1.  [x] Install packages:
    - `uv` project manager.
      ```sh
      pip install uv
      ```
    - `langgraph` and dependencies for lessons.
      ```
      uv pip install -r requirements.txt
      ```
1.  [x] Create an OpenAI account and API key.
1.  [x] Create Tavily account and API Key.
1.  [x] Set up the environment variable.
    ```sh
    echo 'OPENAI_API_KEY=XXXX-XXXX' >> ~/.env
    echo 'TAVILY_API_KEY=XXXX-XXXX' >> ~/.env
    ```
1.  [x] Create a Deep Learning account and complete **AI Agents in LangGraph:**
    - [Video introduction](https://learn.deeplearning.ai/courses/ai-agents-in-langgraph/lesson/1/introduction)
    - [Example 6: Essay Writer](https://learn.deeplearning.ai/courses/ai-agents-in-langgraph/lesson/7/essay-writer)

## Setup

In [1]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage
from langchain_openai import ChatOpenAI

_ = load_dotenv()
    
memory = SqliteSaver.from_conn_string(':memory:')
model = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)

In [2]:
from langchain_core.pydantic_v1 import BaseModel

class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int
    
class Queries(BaseModel):
    queries: List[str]

/home/hanggrian/GitHub/IIT-CS587/langgraph-introduction/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
from textwrap import dedent 

PLAN_PROMPT = \
    dedent(
        '''\
        You are an expert writer tasked with writing a high level outline of an essay.
        Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes
        or instructions for the sections.''',
      )
WRITER_PROMPT = \
    dedent(
        """\
        You are an essay assistant tasked with writing excellent 5-paragraph essays.
        Generate the best essay possible for the user's request and the initial outline.
        If the user provides critique, respond with a revised version of your previous attempts.
        Utilize all the information below as needed:

        ------

        {content}""",
    )
REFLECTION_PROMPT = \
    dedent(
        """\
        You are a teacher grading an essay submission.
        Generate critique and recommendations for the user's submission.
        Provide detailed recommendations, including requests for length, depth, style, etc.""",
    )
RESEARCH_PLAN_PROMPT = \
    dedent(
        '''\
        You are a researcher charged with providing information that can
        be used when writing the following essay. Generate a list of search queries that will gather
        any relevant information. Only generate 3 queries max.''',
    )
RESEARCH_CRITIQUE_PROMPT = \
    dedent(
        '''\
        You are a researcher charged with providing information that can
        be used when making any requested revisions (as outlined below).
        Generate a list of search queries that will gather any relevant information. Only generate 3 queries max.''',
    )

In [4]:
import tavily as tv
import os

tavily = tv.TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task']),
    ]
    response = model.invoke(messages)
    return {'plan': response.content}

def research_plan_node(state: AgentState):
    queries = \
        model.with_structured_output(Queries).invoke([
            SystemMessage(content=RESEARCH_PLAN_PROMPT),
            HumanMessage(content=state['task']),
        ])
    content = state['content'] or []
    for q in queries.queries:
        response = tv.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {'content': content}

def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(content=WRITER_PROMPT.format(content=content)),
        user_message,
    ]
    response = model.invoke(messages)
    return {
        'draft': response.content, 
        'revision_number': state.get('revision_number', 1) + 1,
    }

def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT), 
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {'critique': response.content}

def research_critique_node(state: AgentState):
    queries = \
        model.with_structured_output(Queries).invoke([
            SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
            HumanMessage(content=state['critique']),
        ])
    content = state['content'] or []
    for q in queries.queries:
        response = tv.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {'content': content}

def should_continue(state):
    if state['revision_number'] > state['max_revisions']:
        return END
    return 'reflect'

In [5]:
builder = StateGraph(AgentState)

builder.add_node('planner', plan_node)
builder.add_node('generate', generation_node)
builder.add_node('reflect', reflection_node)
builder.add_node('research_plan', research_plan_node)
builder.add_node('research_critique', research_critique_node)

builder.set_entry_point('planner')

builder.add_conditional_edges(
    'generate', 
    should_continue, 
    {END: END, 'reflect': 'reflect'},
)

builder.add_edge('planner', 'research_plan')
builder.add_edge('research_plan', 'generate')
builder.add_edge('reflect', 'research_critique')
builder.add_edge('research_critique', 'generate')

In [6]:
from IPython.display import Image

graph = builder.compile(checkpointer=memory)
Image(graph.get_graph().draw_png())

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [ ]:
thread = {'configurable': {'thread_id': '1'}}
for s in graph.stream(
    {
        'task': 'what is the difference between langchain and langsmith',
        'max_revisions': 2,
        'revision_number': 1,
    }, 
    thread,
):
    print(s)

## Essay Writer Interface

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from helper import ewriter, writer_gui

In [ ]:
MultiAgent = ewriter()
app = writer_gui(MultiAgent.graph)
app.launch()